In [1]:
# ==================== COMPLETE FEATURE ENGINEERING & MODEL COMPARISON ====================
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import matplotlib.pyplot as plt
import pandas as pd
import gc

In [4]:
# ==================== CLEAR MEMORY ====================
tf.keras.backend.clear_session()
gc.collect()

0

In [5]:
# ==================== PARAMETERS ====================
SAVE_PATH = "windowed_data"
WINDOW_SIZE = 24
HORIZON = 1
EPOCHS = 10
NUM_BATCHES = 5
RANDOM_STATE = 42

In [9]:
# ==================== 1. LOAD DATA ====================
X_all = []
y_all = []

for b in range(NUM_BATCHES):
    X_batch = np.load(os.path.join(SAVE_PATH, f"X_batch_{b}.npy"))
    y_batch = np.load(os.path.join(SAVE_PATH, f"y_batch_{b}.npy"))
    X_all.append(X_batch)
    y_all.append(y_batch)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print(f"Total samples loaded: {len(X_all)}")
print(f"Shape: X={X_all.shape}, y={y_all.shape}")

Total samples loaded: 164424
Shape: X=(164424, 24), y=(164424, 1)


In [10]:
# ==================== 2. SPLIT DATA: 70% TRAIN, 15% VAL, 15% TEST ====================
n_samples = len(X_all)
train_size = int(0.70 * n_samples)
val_size = int(0.15 * n_samples)

X_train_flat = X_all[:train_size]
y_train = y_all[:train_size].flatten()  # FLATTEN HERE

X_val_flat = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size].flatten()  # FLATTEN HERE

X_test_flat = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:].flatten()  # FLATTEN HERE

print(f"Train: {len(X_train_flat)} samples ({len(X_train_flat)/n_samples*100:.1f}%)")
print(f"Val:   {len(X_val_flat)} samples ({len(X_val_flat)/n_samples*100:.1f}%)")
print(f"Test:  {len(X_test_flat)} samples ({len(X_test_flat)/n_samples*100:.1f}%)")

del X_all, y_all
gc.collect()

Train: 115096 samples (70.0%)
Val:   24663 samples (15.0%)
Test:  24665 samples (15.0%)


1158

In [11]:
# ==================== 3. FEATURE ENGINEERING FUNCTIONS ====================
def create_manual_features(X):
    """Create hand-crafted statistical features"""
    features = []
    features.append(np.mean(X, axis=1))
    features.append(np.std(X, axis=1))
    features.append(np.min(X, axis=1))
    features.append(np.max(X, axis=1))
    features.append(X[:, -1])
    features.append(X[:, -1] - X[:, 0])
    features.append(np.median(X, axis=1))
    features.append(np.percentile(X, 25, axis=1))
    features.append(np.percentile(X, 75, axis=1))
    return np.column_stack(features)

def apply_fisher_score(X_train, y_train, X_val, X_test, k=10):
    """Fisher Score (ANOVA F-test) feature selection"""
    selector = SelectKBest(score_func=f_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

def apply_mrmr(X_train, y_train, X_val, X_test, k=10):
    """mRMR approximation using Mutual Information"""
    selector = SelectKBest(score_func=mutual_info_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

def apply_pca(X_train, X_val, X_test, n_components=10):
    """PCA dimensionality reduction"""
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)
    print(f"  PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    return X_train_pca, X_val_pca, X_test_pca



In [12]:
# ==================== 4. BUILD & TRAIN LSTM MODEL ====================
def build_and_train_lstm(X_train, y_train, X_val, y_val, model_name, input_dim):
    """Build and train LSTM model with memory efficiency"""
    print(f"\n  Training {model_name}...")
    
    # Clear previous model from memory
    tf.keras.backend.clear_session()
    gc.collect()
    
    # Reshape for LSTM if needed
    if X_train.ndim == 2:
        X_train = X_train[..., np.newaxis]
        X_val = X_val[..., np.newaxis]
    
    model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(1)
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    
    # Train with SMALL batch size for memory
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=64,  # Small batch size to avoid freeze
        callbacks=[early_stop],
        verbose=1,  # Show progress so you know it's not frozen
        shuffle=True
    )
    
    print(f"  ✓ {model_name} trained (epochs: {len(history.history['loss'])})")
    gc.collect()
    
    return model, history

In [13]:
# ==================== 5. EVALUATION FUNCTION ====================
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model in batches to save memory"""
    if X_test.ndim == 2:
        X_test = X_test[..., np.newaxis]
    
    # Predict in small batches
    y_pred = model.predict(X_test, verbose=0, batch_size=64).reshape(-1)
    
    errors = y_test - y_pred
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    
    delta = 15
    accuracy = (np.sum(np.abs(errors) <= delta) / len(errors)) * 100
    
    return {
        'Model': model_name,
        'MSE':  mse,
        'RMSE': rmse,
        'MAE': mae,
        'Accuracy (%)': accuracy
    }

In [14]:
# ==================== 6. BASELINE MODEL (NAIVE PREDICTION) ====================

y_naive = X_test_flat[:, -1]
errors_naive = y_test - y_naive
mse_naive = np.mean(errors_naive ** 2)
rmse_naive = np.sqrt(mse_naive)
mae_naive = np.mean(np.abs(errors_naive))

delta = 15
correct_count = np.sum(np.abs(errors_naive) <= delta)
total_count = len(errors_naive)
acc_naive = (correct_count / total_count) * 100

baseline_results = {
    'Model': 'Baseline (Naive)',
    'MSE': mse_naive,
    'RMSE': rmse_naive,
    'MAE': mae_naive,
    'Accuracy (%)': acc_naive
}

print(f"Baseline RMSE: {rmse_naive:.2f} mg/dL")
print(f"Baseline MAE: {mae_naive:.2f} mg/dL")
print(f"Baseline Accuracy: {acc_naive:.2f}%")

del y_naive, errors_naive
gc.collect()

Baseline RMSE: 14.75 mg/dL
Baseline MAE: 9.72 mg/dL
Baseline Accuracy: 80.90%


0

In [15]:
# DEBUG: Check shapes
print("\n=== DEBUG INFO ===")
print(f"X_test_flat shape: {X_test_flat.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"y_test first 10 values: {y_test[:10]}")
print(f"y_test min: {y_test.min()}, max: {y_test.max()}")
print("==================\n")


=== DEBUG INFO ===
X_test_flat shape: (24665, 24)
y_test shape: (24665,)
y_test first 10 values: [195.  82. 169. 270. 325. 262. 201. 165. 217. 226.]
y_test min: 40.0, max: 400.0



In [17]:
# ==================== 7. MODEL 1: RAW FEATURES (NO FEATURE SELECTION) ====================
X_train_raw = X_train_flat.copy()
X_val_raw = X_val_flat.copy()
X_test_raw = X_test_flat.copy()

model_raw, hist_raw = build_and_train_lstm(X_train_raw, y_train, X_val_raw, y_val, 
                                            "Raw Features", X_train_raw.shape[1])
results_raw = evaluate_model(model_raw, X_test_raw, y_test, "Raw Features")
print(f"Raw RMSE: {results_raw['RMSE']:.2f} mg/dL, Accuracy: {results_raw['Accuracy (%)']:.2f}%")

del X_train_raw, X_val_raw, X_test_raw, model_raw
gc.collect()


  Training Raw Features...
Epoch 1/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - loss: 17331.3516 - val_loss: 6372.2588
Epoch 2/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 5087.9639 - val_loss: 1462.8531
Epoch 3/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 1612.3558 - val_loss: 475.7066
Epoch 4/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 681.0743 - val_loss: 249.7850
Epoch 5/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 327.0392 - val_loss: 173.6642
Epoch 6/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 213.5719 - val_loss: 165.4474
Epoch 7/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 182.1847 - val_loss: 153.3220
Epoch 8/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 154.6114 - val_loss: 140.0855
Epoch 9/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 146.7891 - val_loss: 127.9380
Epoch 10/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 137.0042 - val_loss: 129.0269
  ✓ Raw Features trained (epochs: 

331

In [18]:
# ==================== 8. MODEL 2: MANUAL FEATURES ====================
X_train_manual = create_manual_features(X_train_flat)
X_val_manual = create_manual_features(X_val_flat)
X_test_manual = create_manual_features(X_test_flat)

print(f"  Manual features created: {X_train_manual.shape[1]} features")

model_manual, hist_manual = build_and_train_lstm(X_train_manual, y_train, X_val_manual, y_val,
                                                  "Manual Features", X_train_manual.shape[1])
results_manual = evaluate_model(model_manual, X_test_manual, y_test, "Manual Features")
print(f"Manual RMSE: {results_manual['RMSE']:.2f} mg/dL, Accuracy: {results_manual['Accuracy (%)']:.2f}%")

del X_train_manual, X_val_manual, X_test_manual, model_manual
gc.collect()

  Manual features created: 9 features

  Training Manual Features...
Epoch 1/10


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 18115.1016 - val_loss: 7079.1567
Epoch 2/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 6031.2549 - val_loss: 1849.7175
Epoch 3/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 1981.7858 - val_loss: 692.6149
Epoch 4/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 867.5643 - val_loss: 419.5414
Epoch 5/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 517.3197 - val_loss: 360.5671
Epoch 6/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 402.4121 - val_loss: 321.2747
Epoch 7/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 366.8281 - val_loss: 327.9504
Epoch 8/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 358.4359 - val_loss: 330.8597
Epoch 9/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 348.4799 - val_loss: 312.8286
Epoch 10/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 346.2865 - val_loss: 359.0461
  ✓ Manual Features trained (epochs: 10)
Manual RMSE: 15.37 mg/dL, Accuracy: 78.08%


331

In [20]:
# ==================== 9. MODEL 3: FISHER SCORE ====================
K_FISHER = 10
X_train_fisher, X_val_fisher, X_test_fisher = apply_fisher_score(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_FISHER
)
print(f"  Fisher: Selected top {K_FISHER} features")

model_fisher, hist_fisher = build_and_train_lstm(X_train_fisher, y_train, X_val_fisher, y_val,
                                                  "Fisher Score", X_train_fisher.shape[1])
results_fisher = evaluate_model(model_fisher, X_test_fisher, y_test, "Fisher Score")
print(f"Fisher RMSE: {results_fisher['RMSE']:.2f} mg/dL, Accuracy: {results_fisher['Accuracy (%)']:.2f}%")

del X_train_fisher, X_val_fisher, X_test_fisher, model_fisher
gc.collect()

  Fisher: Selected top 10 features

  Training Fisher Score...
Epoch 1/10


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 17335.0312 - val_loss: 6384.4829
Epoch 2/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 4893.6162 - val_loss: 1396.0636
Epoch 3/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 1576.0208 - val_loss: 473.1189
Epoch 4/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 623.6073 - val_loss: 233.7150
Epoch 5/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 300.7849 - val_loss: 160.9559
Epoch 6/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 190.2891 - val_loss: 133.0483
Epoch 7/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 153.7939 - val_loss: 120.2258
Epoch 8/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 137.0075 - val_loss: 119.3435
Epoch 9/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 129.5765 - val_loss: 117.9692
Epoch 10/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 125.5681 - val_loss: 111.5237
  ✓ Fisher Score trained (epochs: 10)
Fisher RMSE: 11.18 mg/dL, Accuracy: 90.57%


331

In [21]:
# ==================== 10. MODEL 4: Mutual mRMR ====================
K_MRMR = 10
X_train_mrmr, X_val_mrmr, X_test_mrmr = apply_mrmr(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_MRMR
)
print(f"  mRMR: Selected top {K_MRMR} features")

model_mrmr, hist_mrmr = build_and_train_lstm(X_train_mrmr, y_train, X_val_mrmr, y_val,
                                              "mRMR", X_train_mrmr.shape[1])
results_mrmr = evaluate_model(model_mrmr, X_test_mrmr, y_test, "mRMR")
print(f"mRMR RMSE: {results_mrmr['RMSE']:.2f} mg/dL, Accuracy: {results_mrmr['Accuracy (%)']:.2f}%")

del X_train_mrmr, X_val_mrmr, X_test_mrmr, model_mrmr
gc.collect()

  mRMR: Selected top 10 features

  Training mRMR...
Epoch 1/10


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 16226.0889 - val_loss: 5460.5293
Epoch 2/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 4496.4072 - val_loss: 1323.0054
Epoch 3/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 1514.1740 - val_loss: 449.6792
Epoch 4/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 600.3740 - val_loss: 227.1338
Epoch 5/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 291.7070 - val_loss: 154.2197
Epoch 6/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 187.8568 - val_loss: 135.9144
Epoch 7/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 156.0242 - val_loss: 127.8069
Epoch 8/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 138.8424 - val_loss: 115.8231
Epoch 9/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 133.3040 - val_loss: 114.5514
Epoch 10/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 128.6579 - val_loss: 114.9885
  ✓ mRMR trained (epochs: 10)
mRMR RMSE: 11.34 mg/dL, Accuracy: 89.86%


331

In [22]:
# ==================== 11. MODEL 5: PCA ====================
N_COMPONENTS = 10
X_train_pca, X_val_pca, X_test_pca = apply_pca(
    X_train_flat, X_val_flat, X_test_flat, n_components=N_COMPONENTS
)
print(f"  PCA: Reduced to {N_COMPONENTS} components")

model_pca, hist_pca = build_and_train_lstm(X_train_pca, y_train, X_val_pca, y_val,
                                            "PCA", X_train_pca.shape[1])
results_pca = evaluate_model(model_pca, X_test_pca, y_test, "PCA")
print(f"PCA RMSE: {results_pca['RMSE']:.2f} mg/dL, Accuracy: {results_pca['Accuracy (%)']:.2f}%")

del X_train_pca, X_val_pca, X_test_pca, model_pca
gc.collect()

  PCA variance explained: 99.18%
  PCA: Reduced to 10 components

  Training PCA...
Epoch 1/10


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 15690.1133 - val_loss: 5283.3345
Epoch 2/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 5185.5752 - val_loss: 2013.6500
Epoch 3/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 2061.2761 - val_loss: 805.6465
Epoch 4/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 918.9333 - val_loss: 474.6491
Epoch 5/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 568.2599 - val_loss: 426.0739
Epoch 6/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 445.5356 - val_loss: 359.9132
Epoch 7/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 387.6720 - val_loss: 335.4660
Epoch 8/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 361.1990 - val_loss: 314.3299
Epoch 9/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 342.9033 - val_loss: 298.3855
Epoch 10/10
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 333.0679 - val_loss: 323.7572
  ✓ PCA trained (epochs: 10)
PCA RMSE: 17.69 mg/dL, Accuracy: 71.30%


331

In [23]:
# ==================== 12. FINAL COMPARISON ====================
# Compile all results
all_results = [
    baseline_results,
    results_raw,
    results_manual,
    results_fisher,
    results_mrmr,
    results_pca
]

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('RMSE')

print("\n" + results_df.to_string(index=False))

# Find best model
best_model = results_df.iloc[0]['Model']
best_rmse = results_df.iloc[0]['RMSE']
best_acc = results_df.iloc[0]['Accuracy (%)']

print(f"BEST MODEL: {best_model}")
print(f"RMSE: {best_rmse:.2f} mg/dL")
print(f"Accuracy: {best_acc:.2f}%")


           Model        MSE      RMSE       MAE  Accuracy (%)
    Fisher Score 124.959152 11.178513  6.766038     90.565579
            mRMR 128.691254 11.344216  6.754711     89.860126
    Raw Features 133.449539 11.552036  7.000118     89.774985
Baseline (Naive) 217.547867 14.749504  9.720819     80.904115
 Manual Features 236.268463 15.371027 10.351640     78.082303
             PCA 312.850830 17.687590 12.301910     71.295358
BEST MODEL: Fisher Score
RMSE: 11.18 mg/dL
Accuracy: 90.57%


In [24]:
# ==================== 13. VISUALIZATIONS ====================

# Plot 1: RMSE Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE Bar Chart
axes[0].bar(results_df['Model'], results_df['RMSE'], color='steelblue')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('RMSE (mg/dL)', fontsize=12)
axes[0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Accuracy Bar Chart
axes[1].bar(results_df['Model'], results_df['Accuracy (%)'], color='coral')
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Comparison (±15 mg/dL)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# MAE Bar Chart
axes[2].bar(results_df['Model'], results_df['MAE'], color='seagreen')
axes[2].set_xlabel('Model', fontsize=12)
axes[2].set_ylabel('MAE (mg/dL)', fontsize=12)
axes[2].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: model_comparison.png")
plt.close()

# Plot 2: Training History
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2 = axes2.flatten()

histories = [
    (hist_raw, 'Raw Features'),
    (hist_manual, 'Manual Features'),
    (hist_fisher, 'Fisher Score'),
    (hist_mrmr, 'mRMR'),
    (hist_pca, 'PCA')
]

for idx, (hist, name) in enumerate(histories):
    axes2[idx].plot(hist.history['loss'], label='Train Loss', linewidth=2)
    axes2[idx].plot(hist.history['val_loss'], label='Val Loss', linewidth=2)
    axes2[idx].set_xlabel('Epoch', fontsize=10)
    axes2[idx].set_ylabel('Loss (MSE)', fontsize=10)
    axes2[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes2[idx].legend()
    axes2[idx].grid(True, alpha=0.3)

axes2[5].axis('off')

plt.tight_layout()
plt.savefig('training_histories.png', dpi=300, bbox_inches='tight')
print("Saved: training_histories.png")
plt.close()

Saved: model_comparison.png
Saved: training_histories.png


In [25]:
# ==================== 14. SAVE RESULTS ====================
results_df.to_csv('model_comparison_results.csv', index=False)
print("Saved: model_comparison_results.csv")

# Final cleanup
del X_train_flat, X_val_flat, X_test_flat, y_train, y_val, y_test
gc.collect()

Saved: model_comparison_results.csv


51